# Свёрточные нейросети

## 1. Зачем нужны свёрточные сети

Полносвязная сеть воспринимает картинку как длинный список чисел. Для нее пиксель в левом верхнем углу и пиксель рядом с ним не являются «соседями» особым образом: это просто разные координаты вектора.

CNN использует факт, что изображения имеют структуру:

- соседние пиксели часто связаны;
- один и тот же объект может немного сдвинуться;
- полезные признаки часто локальны: границы, углы, текстуры, части объекта;
- один фильтр можно применять ко всему изображению, а не учить отдельные веса для каждого места.

Главная идея: маленький фильтр скользит по изображению и ищет один и тот же паттерн в разных местах.


## 2. Изображение как тензор

Чёрно-белое изображение можно представить как матрицу: высота x ширина.

Цветное RGB-изображение — это три матрицы:

- R: интенсивность красного канала;
- G: интенсивность зелёного канала;
- B: интенсивность синего канала.

В глубоком обучении такое изображение обычно хранится как тензор с формой:

- PyTorch: channels x height x width;
- TensorFlow/Keras: height x width x channels.

Для батча добавляется первое измерение batch_size.


**Рис. 3. RGB-изображение как три канала**

![Рис. 3. RGB-изображение как три канала](https://yastatic.net/s3/education-portal/media/2_rgb_rooster_1_9610445be0_11cb0f9215.svg)

**Рис. 4. Разделение изображения на RGB-каналы**

![Рис. 4. Разделение изображения на RGB-каналы](https://yastatic.net/s3/education-portal/media/1_rgb_split_rooster_1_72ea43ba3f_b7355dedf3.svg)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = False

# Игрушечное "изображение": 5 x 5 пикселей, один канал.
image = np.array([
    [0, 0, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
], dtype=float)

plt.imshow(image, cmap="gray")
plt.title("Игрушечное изображение 5 x 5")
plt.colorbar()
plt.show()


## 3. Сдвиги, масштаб, повороты и padding

В реальных данных объект редко стоит строго в одном месте и в одном масштабе.

CNN полезны, потому что один и тот же фильтр проходит по разным областям изображения. Если фильтр научился находить «край уха» или «вертикальную границу», он может найти этот признак в разных местах.

Важно различать две идеи:

- эквивариантность к сдвигу: если вход сдвинулся, карта признаков тоже сдвинулась;
- инвариантность: итоговый ответ модели почти не меняется при небольшом сдвиге объекта.

CNN сама по себе ближе к эквивариантности. Инвариантность обычно усиливают pooling, global pooling, augmentation и обучением на разных вариантах изображений.


**Рис. 5. Исходное изображение для примеров преобразований**

![Рис. 5. Исходное изображение для примеров преобразований](https://yastatic.net/s3/education-portal/media/puppy_a1d259493d_708f608dce.webp)

**Рис. 6. Сдвиг изображения**

![Рис. 6. Сдвиг изображения](https://yastatic.net/s3/education-portal/media/shifted_puppy_c27db8708f_ca16064a3b.webp)

**Рис. 7. Масштабирование изображения**

![Рис. 7. Масштабирование изображения](https://yastatic.net/s3/education-portal/media/scaled_puppy_d6c7abac44_a80f12b60e.webp)

**Рис. 8. Поворот изображения**

![Рис. 8. Поворот изображения](https://yastatic.net/s3/education-portal/media/rotated_puppy_523e9847b2_31bef7bb3c.webp)

**Рис. 9. Отражение изображения**

![Рис. 9. Отражение изображения](https://yastatic.net/s3/education-portal/media/flipped_puppy_a8a6029dce_c8300e6bd9.webp)


### Padding

Когда фильтр подходит к краю изображения, ему не хватает соседних пикселей за границей. Padding добавляет рамку вокруг изображения.

Чаще всего используют нули. Это позволяет:

- не терять информацию на краях;
- сохранить размер карты признаков при stride=1;
- строить более глубокие сети, где размер не уменьшается слишком быстро.

Без padding карта признаков становится меньше. С padding можно сделать так, чтобы высота и ширина остались прежними.


**Рис. 10. Идея сдвига окна/фильтра**

![Рис. 10. Идея сдвига окна/фильтра](https://yastatic.net/s3/education-portal/media/gif_shift_template_35384897e9_16cdab70dc.svg)

**Рис. 11. Padding: добавляем рамку вокруг изображения**

![Рис. 11. Padding: добавляем рамку вокруг изображения](https://yastatic.net/s3/education-portal/media/padded_puppy_3bd370a19b_83e0aba93a.webp)


In [ ]:
def add_zero_padding(x: np.ndarray, pad: int) -> np.ndarray:
    return np.pad(x, pad_width=pad, mode="constant", constant_values=0)

padded = add_zero_padding(image, pad=1)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title(f"Исходное: {image.shape}")
axes[1].imshow(padded, cmap="gray")
axes[1].set_title(f"С padding=1: {padded.shape}")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()


## 4. Свёртка: фильтр скользит по изображению

Свёрточный фильтр часто называют kernel. Например, фильтр 3 x 3 смотрит на маленький квадрат из 9 пикселей.

Алгоритм:

1. Берём окно изображения такого же размера, как фильтр.
2. Умножаем каждую клетку окна на соответствующую клетку фильтра.
3. Складываем результаты.
4. Записываем одно число в карту признаков.
5. Сдвигаем окно и повторяем.

Фильтр не задаётся человеком навсегда. В CNN значения фильтра — обучаемые параметры, которые модель подбирает по данным.


**Рис. 12. Свёртка как движение фильтра**

![Рис. 12. Свёртка как движение фильтра](https://yastatic.net/s3/education-portal/media/gif_conv_template0_fbf83244f0_c38899448f.svg)

**Рис. 13. Свёртка по тензору изображения**

![Рис. 13. Свёртка по тензору изображения](https://yastatic.net/s3/education-portal/media/11_image_tensor_conv_cdaf6d395d_f54b12323d_a3d8bf4b65.svg)

**Рис. 14. Примеры фильтров и карт признаков**

![Рис. 14. Примеры фильтров и карт признаков](https://yastatic.net/s3/education-portal/media/conv_examples_7a0dba822d_10b47a420f.webp)

**Рис. 15. Маленький пример свёртки**

![Рис. 15. Маленький пример свёртки](https://yastatic.net/s3/education-portal/media/gif_small_conv_template_a2dd681565_6ce335cf1b.svg)

**Рис. 16. Результат маленькой свёртки**

![Рис. 16. Результат маленькой свёртки](https://yastatic.net/s3/education-portal/media/very_small_conv_4bdde25018_dab7fdba0b.webp)


In [ ]:
def conv2d_valid(x: np.ndarray, kernel: np.ndarray, stride: int = 1) -> np.ndarray:
    """Мини-реализация 2D-свёртки без padding для одного канала."""
    kh, kw = kernel.shape
    out_h = (x.shape[0] - kh) // stride + 1
    out_w = (x.shape[1] - kw) // stride + 1
    output = np.zeros((out_h, out_w), dtype=float)

    for i in range(out_h):
        for j in range(out_w):
            row = i * stride
            col = j * stride
            window = x[row:row + kh, col:col + kw]
            output[i, j] = np.sum(window * kernel)

    return output

# Фильтр, который реагирует на вертикальные перепады.
vertical_edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

conv_result = conv2d_valid(image, vertical_edge_kernel, stride=1)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Вход")
axes[1].imshow(vertical_edge_kernel, cmap="coolwarm")
axes[1].set_title("Фильтр")
axes[2].imshow(conv_result, cmap="coolwarm")
axes[2].set_title("Карта признаков")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

print("input shape:", image.shape)
print("kernel shape:", vertical_edge_kernel.shape)
print("output shape:", conv_result.shape)
print(conv_result)


### Размер выхода

Если padding нет, а фильтр 3 x 3, то он не может встать в самый край как центр. Поэтому изображение уменьшается.

Практическое правило:

- valid convolution: без рамки, выход меньше входа;
- same convolution: с рамкой, при stride=1 выход примерно того же размера;
- stride=2: окно прыгает через один пиксель, выход примерно в два раза меньше.

Если всё же нужна формула для одного измерения:

    output_size = floor((input_size + 2 * padding - kernel_size) / stride) + 1

In [ ]:
def conv_output_size(input_size: int, kernel_size: int, padding: int = 0, stride: int = 1) -> int:
    return (input_size + 2 * padding - kernel_size) // stride + 1

examples = [
    {"input": 32, "kernel": 3, "padding": 0, "stride": 1},
    {"input": 32, "kernel": 3, "padding": 1, "stride": 1},
    {"input": 32, "kernel": 3, "padding": 1, "stride": 2},
    {"input": 28, "kernel": 5, "padding": 0, "stride": 1},
]

for ex in examples:
    out = conv_output_size(ex["input"], ex["kernel"], ex["padding"], ex["stride"])
    print(f"input={ex['input']}, kernel={ex['kernel']}, padding={ex['padding']}, stride={ex['stride']} -> output={out}")


## 5. Много фильтров, каналы и карты признаков

Один фильтр ищет один тип локального паттерна. Например:

- вертикальные границы;
- горизонтальные границы;
- углы;
- простые текстуры.

В реальной CNN на слое обычно десятки или сотни фильтров. Каждый фильтр создаёт свою карту признаков. Если фильтров 32, на выходе получится 32 канала.

Для RGB-входа фильтр тоже имеет глубину 3: он смотрит сразу на R, G и B. Поэтому в PyTorch слой записывают так:

    nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)

Это значит: на входе 3 канала, на выходе 32 карты признаков.


## 6. Глубина сети и receptive field

Один слой 3 x 3 видит только маленький участок изображения. Но если поставить несколько свёрточных слоёв подряд, каждый следующий слой видит признаки, собранные предыдущим.

Простая интуиция:

- ранние слои ищут линии и края;
- средние слои ищут текстуры и части объектов;
- поздние слои собирают более крупные и смысловые признаки.

Receptive field — это область исходного изображения, которая влияет на один нейрон в глубоком слое.

Dilated convolution делает «дырки» в фильтре: фильтр смотрит на более широкую область без сильного роста числа параметров.


**Рис. 17. Композиция нескольких свёрток**

![Рис. 17. Композиция нескольких свёрток](https://yastatic.net/s3/education-portal/media/15_conv_composition_e686591745_b5d7657ff6_01f88a670a.svg)

**Рис. 18. Receptive field: какую область входа видит нейрон**

![Рис. 18. Receptive field: какую область входа видит нейрон](https://yastatic.net/s3/education-portal/media/16_receptive_field_fbd1c12519_af58719882_dc46e45295.svg)

**Рис. 19. Dilated convolution: свёртка с пропусками**

![Рис. 19. Dilated convolution: свёртка с пропусками](https://yastatic.net/s3/education-portal/media/dilated_convolution_36bafd970d.webp)

**Рис. 20. Определение свёрточного слоя**

![Рис. 20. Определение свёрточного слоя](https://yastatic.net/s3/education-portal/media/18_conv_def_142fec410a_2b151b5c5f_23ec612516.svg)


In [ ]:
def receptive_field(kernel_sizes):
    """Упрощенный расчет receptive field для stride=1 и без dilation."""
    rf = 1
    values = []
    for k in kernel_sizes:
        rf += k - 1
        values.append(rf)
    return values

print("Три слоя 3x3 дают receptive field:", receptive_field([3, 3, 3]))
print("Пять слоев 3x3 дают receptive field:", receptive_field([3, 3, 3, 3, 3]))
print("Один слой 7x7 дает receptive field:", receptive_field([7]))

# Вывод для обсуждения:
# несколько маленьких сверток часто гибче, чем одна большая, потому что между ними есть нелинейности.


## 7. Свёртки не только для изображений

Свёртку можно применять к любой структуре, где есть локальное соседство.

Для текста 1D-свёртка может смотреть на короткие фрагменты последовательности:

- 2-3 соседних слова;
- несколько символов;
- участок временного ряда.

Сейчас для NLP чаще используют Transformers, но 1D-CNN остаются полезными для быстрых моделей, сигналов и временных рядов.


**Рис. 21. Свёртка для последовательности слов: схема 1**

![Рис. 21. Свёртка для последовательности слов: схема 1](https://yastatic.net/s3/education-portal/media/cnn_word_1_a7eae439ca_e7becfd688.svg)

**Рис. 22. Свёртка для последовательности слов: схема 2**

![Рис. 22. Свёртка для последовательности слов: схема 2](https://yastatic.net/s3/education-portal/media/cnn_word_c294cce45a_13a54daa9b.svg)


## 8. Pooling и global pooling

Pooling уменьшает пространственный размер карты признаков.

Max pooling берёт максимум в маленьком окне. Интуитивно это значит: «если признак где-то рядом найден, оставим самый сильный отклик».

Зачем pooling:

- уменьшить размер данных внутри сети;
- снизить вычисления;
- сделать модель устойчивее к небольшим сдвигам;
- оставить наиболее заметные признаки.

Global pooling берёт среднее или максимум сразу по всей карте признаков. Это удобный способ перейти от карт признаков к вектору для классификации.


**Рис. 23. Max pooling**

![Рис. 23. Max pooling](https://yastatic.net/s3/education-portal/media/21_maxpool_b19a0a26b2_dbd86d51dd_acdf90632c.svg)

**Рис. 24. Свёртка/пулинг как скользящее окно**

![Рис. 24. Свёртка/пулинг как скользящее окно](https://yastatic.net/s3/education-portal/media/gif_conv_template1_5c2a2c90c6_8f14127d42.svg)

**Рис. 25. Global pooling**

![Рис. 25. Global pooling](https://yastatic.net/s3/education-portal/media/23_globalpool_8b3ea2ca37_797f26c48e_55f0fc2755.svg)


In [ ]:
def max_pool2d(x: np.ndarray, kernel_size: int = 2, stride: int = 2) -> np.ndarray:
    out_h = (x.shape[0] - kernel_size) // stride + 1
    out_w = (x.shape[1] - kernel_size) // stride + 1
    output = np.zeros((out_h, out_w), dtype=float)

    for i in range(out_h):
        for j in range(out_w):
            row = i * stride
            col = j * stride
            window = x[row:row + kernel_size, col:col + kernel_size]
            output[i, j] = np.max(window)

    return output

pooled = max_pool2d(add_zero_padding(image, 1), kernel_size=2, stride=2)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(add_zero_padding(image, 1), cmap="gray")
axes[0].set_title("До pooling")
axes[1].imshow(pooled, cmap="gray")
axes[1].set_title("После max pooling")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

print("before:", add_zero_padding(image, 1).shape)
print("after:", pooled.shape)
print(pooled)


## 9. Residual connections

Residual connection добавляет короткий путь вокруг нескольких слоёв.

Интуиция:

- обычный блок должен выучить всю новую функцию целиком;
- residual-блок может выучить только поправку к входу;
- если блок не нужен, сети легче научиться почти ничего не менять.

Это помогает обучать очень глубокие CNN. Самая известная архитектура с такой идеей — ResNet.


**Рис. 26. Residual connection**

![Рис. 26. Residual connection](https://yastatic.net/s3/education-portal/media/24_residual_bb9168282a_b20cdede5b_7417348fb7.svg)



## 10. Простой CNN pipeline в PyTorch

Ниже пример без пользовательских классов. Он показывает структуру, но не скачивает данные и не запускает обучение автоматически.

Типичный pipeline:

1. Подготовить изображения в формате batch x channels x height x width.
2. Нормализовать пиксели.
3. Создать DataLoader.
4. Собрать модель из Conv2d, ReLU, Pooling, Flatten, Linear.
5. Выбрать loss: обычно CrossEntropyLoss для классификации.
6. Выбрать optimizer: AdamW или SGD.
7. В цикле: forward -> loss -> backward -> optimizer.step.
8. Проверять качество на validation.


In [ ]:
try:
    import torch
    from torch import nn

    simple_cnn = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(32, 10),  # 10 классов
    )

    x = torch.randn(8, 3, 32, 32)  # batch=8, RGB, 32x32
    logits = simple_cnn(x)

    print(simple_cnn)
    print("input:", x.shape)
    print("logits:", logits.shape)

except ModuleNotFoundError:
    print("PyTorch не установлен. Для запуска ячейки выполните: python -m pip install torch")


### Как читать архитектуру выше

- Conv2d(3, 16, 3, padding=1): берём RGB-картинку и строим 16 карт признаков.
- ReLU: добавляем нелинейность.
- MaxPool2d(2): уменьшаем высоту и ширину примерно в 2 раза.
- Conv2d(16, 32, 3, padding=1): строим более богатые признаки.
- AdaptiveAvgPool2d((1, 1)): превращаем каждую карту признаков в одно число.
- Flatten: делаем обычный вектор.
- Linear(32, 10): получаем 10 logits для 10 классов.

Для другого числа классов нужно поменять только последний Linear.
